In [37]:
import pandas as pd
import sweetviz as sv
import numpy as np
import logging
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib


In [2]:
df = pd.read_csv('student_performance_dataset.csv')


In [52]:
report = sv.analyze(df)
# 2. Save and open in your web browser:
report.show_html("eda_report.html")

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)

Report eda_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [3]:
# Configuración de Logging para Airflow
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# Defino lo que espero recibir
expected_features = {
    'student_id':{
        'type':'int64',
        'transform': 'drop', 
    },
    'gender':{
        'type':'category',
        'valid': ['Female', 'Male'],
        'transform': 'ohe'
    },
    'study_time_hours':{
        'type':'float64',
        'transform': 'num',
    },
    'attendance_percent':{
        'type':'float64',
        'transform': 'num',
    },
    'sleep_hours':{
        'type':'float64',
        'transform': 'num',
    },
    'parental_education':{
        'type':'category',
        'valid': ['High School', 'Bachelors', 'Masters', 'PhD'],
        'transform': 'ohe'
    },
    'internet_access':{
        'type':'category',
        'valid': ['Yes', 'No'],
        'transform': 'ohe'
    },
    'extracurricular_activities':{
        'type':'category',
        'valid': ['Yes', 'No'],
        'transform': 'ohe'
    },
    'part_time_job':{
        'type':'category',
        'valid': ['Yes', 'No'],
        'transform': 'ohe'
    },
    'previous_grade':{
        'type':'float64',
        'transform': 'num', 
    },
    'final_exam_score':{
        'type':'float64',
        'transform': 'drop', 
    },
    'final_grade':{
        'type':'category',
        'valid': ['A', 'B', 'C', 'D', 'E', 'F'],
        'transform': 'label'
    }
}

In [5]:
# Reviso que las features uqe llegan son las que espero
expected_set = set(expected_features.keys())
actual_set = set(df.columns)
missing_features = expected_set - actual_set
added_features = actual_set - expected_set
is_exact_set_match = (missing_features == set() and added_features == set())
if is_exact_set_match:
    logger.info(f"Feature names and quantities came as expected")
elif len(missing_features) > 0:
    logger.error(f"Missing features ({len(missing_features)}): {missing_features}")
else:
    logger.error(f"Added/Extra features ({len(added_features)}): {added_features}")

INFO - Feature names and quantities came as expected


In [6]:
#Fuerzo los tipos de datos
dtype_mapping = {
    col: pd.CategoricalDtype(categories=spec['valid']) 
         if spec['type'] == 'category' and 'valid' in spec 
         else spec['type']
    for col, spec in expected_features.items()
}
try:
    df = df.astype(dtype_mapping)
    logger.info("Data types converted")
except:
    logger.error("Cant convert datatypes")

INFO - Data types converted


In [38]:
# CONFIG
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALERT_THRESHOLD = 0.15
IQR_MULTIPLIER = 1.5
UNKNOWN_FILL_VALUE = 'UNKNOWN'

# TRANSFORMADORES
class DataQualityAlerter(BaseEstimator, TransformerMixin):
    """Revisa el % de nulos y emite un warning si supera el umbral."""
    def __init__(self, alert_threshold=0.15):
        self.alert_threshold = alert_threshold

    def fit(self, X, y=None):
        return self # No aprende nada, solo inspecciona

    def transform(self, X):
        missing_ratios = X.isnull().mean()
        for col, ratio in missing_ratios.items():
            if ratio > self.alert_threshold:
                logger.warning(f"ALERTA CALIDAD: La columna '{col}' tiene {ratio:.2%} de nulos "
                               f"(Umbral: {self.alert_threshold:.2%})")
        return X

    def get_feature_names_out(self, input_features=None):
        return input_features

class IQROutlierHandler(BaseEstimator, TransformerMixin):
    """Detecta, alerta (si supera umbral) y recorta (cap/winsorize) outliers usando IQR."""
    def __init__(self, multiplier=1.5, alert_threshold=0.15):
        self.multiplier = multiplier
        self.alert_threshold = alert_threshold
        self.lower_bounds_ = {}
        self.upper_bounds_ = {}

    def fit(self, X, y=None):
        # Aprendemos los límites SOLO con los datos de entrenamiento
        for col in X.columns:
            q1 = X[col].quantile(0.25)
            q3 = X[col].quantile(0.75)
            iqr = q3 - q1
            self.lower_bounds_[col] = q1 - (self.multiplier * iqr)
            self.upper_bounds_[col] = q3 + (self.multiplier * iqr)
        return self

    def transform(self, X):
        X_trans = X.copy()
        for col in X.columns:
            # Calcular % de outliers para la alerta
            is_outlier = (X_trans[col] < self.lower_bounds_[col]) | (X_trans[col] > self.upper_bounds_[col])
            outlier_ratio = is_outlier.mean()
            
            if outlier_ratio > self.alert_threshold:
                logger.warning(f"ALERTA OUTLIERS: La columna '{col}' tiene {outlier_ratio:.2%} de outliers "
                               f"(Umbral: {self.alert_threshold:.2%})")
            
            # Recortar (capping) en lugar de eliminar filas para no romper el shape del dataset
            X_trans[col] = X_trans[col].clip(lower=self.lower_bounds_[col], upper=self.upper_bounds_[col])
        return X_trans
    
    def get_feature_names_out(self, input_features=None):
        return input_features

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Codifica categorías basándose en su frecuencia de aparición."""
    def __init__(self):
        self.mapping_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            # Diccionario con la frecuencia relativa de cada categoría
            self.mapping_[col] = X[col].value_counts(normalize=True).to_dict()
        return self

    def transform(self, X):
        X_trans = X.copy()
        for col in X.columns:
            # Mapea y rellena con 0 si en Test aparece una categoría no vista en Train
            X_trans[col] = X_trans[col].map(self.mapping_[col]).fillna(0)
        return X_trans
    
    def get_feature_names_out(self, input_features=None):
        return input_features


# FUNCION PRINCIPAL DE TRANSFORMACION

def run_etl_transformation(df, target_col, num_cols, cat_ohe_cols, cat_freq_cols, cols_to_drop):
    """
    Ejecuta el pipeline completo. Devuelve DataFrames limpios listos para modelar.
    """
    logger.info("Iniciando tarea de transformación...")

    # 1. Split
    X = df.drop(columns=[target_col])
    y = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    #Unimos todas las columnas categóricas para la fase inicial
    cat_cols = cat_ohe_cols + cat_freq_cols

    #Derivamos segun el tipo de encoder
    categorical_encoders = ColumnTransformer(
        transformers=[
            ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first'), cat_ohe_cols),
            ('freq', FrequencyEncoder(), cat_freq_cols)
        ],
        remainder='drop',
        n_jobs=-1
    )

    #Pipeline para categoricas
    cat_pipeline = Pipeline([
        ('alert_nulls', DataQualityAlerter(alert_threshold=ALERT_THRESHOLD)),
        ('imputer', SimpleImputer(strategy='constant', fill_value=UNKNOWN_FILL_VALUE)),
        ('encoders', categorical_encoders)
    ])

    #Pipeline para numericas
    num_pipeline = Pipeline([
        ('alert_nulls', DataQualityAlerter(alert_threshold=ALERT_THRESHOLD)),
        ('imputer', SimpleImputer(strategy='mean')),
        ('outliers', IQROutlierHandler(multiplier=IQR_MULTIPLIER, alert_threshold=ALERT_THRESHOLD)),
        ('scaler', StandardScaler())
    ])

    # Preprocesador master
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_pipeline, num_cols),
            ('cat', cat_pipeline, cat_cols),
            ('dropper', 'drop', cols_to_drop),
        ],
        remainder='drop',
        n_jobs=-1
    )

    preprocessor.set_output(transform="pandas")

    # Ajustar en Train y Transformar ambos
    logger.info("Ajustando pipeline en datos de entrenamiento...")
    X_train_processed = preprocessor.fit_transform(X_train)
    
    logger.info("Aplicando transformaciones en datos de prueba...")
    X_test_processed = preprocessor.transform(X_test)

    logger.info("Transformación completada con éxito.")
    
    export_path = "etl_preprocessor.joblib"
    joblib.dump(preprocessor, export_path)
    return X_train_processed, X_test_processed, y_train, y_test

In [36]:
target_col = "final_grade"
num_cols = [k for k,v in expected_features.items() if v['transform'] == 'num']
cat_ohe_cols = [k for k,v in expected_features.items() if v['transform'] == 'ohe']
cat_freq_cols = [k for k,v in expected_features.items() if v['transform'] == 'freq']
cols_to_drop = [k for k,v in expected_features.items() if v['transform'] == 'drop']

total_feats = len(num_cols) + len(cat_freq_cols) + len(cat_ohe_cols) + len(cols_to_drop)
if total_feats != len(df.keys())-1:
    logger.error("Some features are not being processed")
else:
    logger.info("All features to be processed")

INFO - All features to be processed


In [39]:
X_train, X_test, y_train, y_test = run_etl_transformation(
    df,
    target_col,
    num_cols,
    cat_ohe_cols,
    cat_freq_cols,
    cols_to_drop
)

INFO - Iniciando tarea de transformación...
INFO - Ajustando pipeline en datos de entrenamiento...
INFO - Aplicando transformaciones en datos de prueba...
INFO - Transformación completada con éxito.


In [20]:
report = sv.analyze(X_train)
# 2. Save and open in your web browser:
report.show_html("eda_report_proc.html")

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)    

Report eda_report_proc.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.
